## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)

2. A function or coroutine to execute.

In [8]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("openai:gpt-4o-mini")
response = model.invoke("hey")
response

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c66758d87c', 'id': 'chatcmpl-Dq15N5VSQ3Wer7QpOUCSjboofjWp9', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebd22-445f-7280-a65e-64962983e1f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [9]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [10]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args:{tool_call['args']}")


content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 51, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0004d2b74a', 'id': 'chatcmpl-Dq15U4UfH67460Iv5I2r4bCGcQKWc', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ebd22-63c3-7250-bbdd-1b4e44be016c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_JxO2HIpoVM48wLq2OdntndMm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 14, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Tool: get_weather
Args:{'lo

### Tool execution loop

In [17]:
# Step 1: model generates tool calls
messages = [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
print(ai_msg)
messages.append(ai_msg)
print(messages)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tools with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    print(messages)

#Step 3: Pass results back to model for the final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72 degree F and sunny"


content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 50, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0004d2b74a', 'id': 'chatcmpl-Dq1EsPCHjAe8JTHKe8HEADlAd89Pk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ebd2b-4303-7672-aa79-50a355147711-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_hULOU5r5GR2Jj2R8i5vHlLiV', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 14, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
[{'role': 'user', 'content'